# OOP in Python — Hands-On Practice

This notebook contains the 8 homework tasks from the OOP practice sheet.

## How to use this notebook
- Read each **problem definition** in the Markdown cell.
- Write your solution **only** between:

```python
# START WRITING HERE
# END HERE
```

- Cells marked `# DO NOT TOUCH` contain tests and checking code.
- Run the student solution cell first, then run its test cell.
- In Google Colab, if definitions get mixed up, use **Runtime → Restart session** and run again from the top.


## Task 1 — BankAccount

Build a small account class that also keeps track of how many accounts exist.

### Requirements
1. Class attributes: `bank_name = "AI Academy Bank"` and `total_accounts = 0`.
2. `__init__(self, owner, balance=0.0)` stores owner and balance, rejects a negative starting balance, increments the shared counter, and assigns `account_id` as `ACC-001`, `ACC-002`, ...
3. `deposit(amount)` rejects non-positive amounts.
4. `withdraw(amount)` rejects non-positive amounts and raises `ValueError` when there is not enough money.
5. `summary()` returns a string such as `[ACC-001] Aysel: 650.00 AZN`.
6. Support the scenario: Aysel starts with 500, Rashad with 0; deposit 250 and withdraw 100 for Aysel; deposit 80 for Rashad.


In [1]:
# START WRITING HERE

class BankAccount:
    bank_name = "AI Academy Bank" 
    total_accounts = 0

    def __init__(self, owner, balance = 0.0):
        if balance < 0:
            raise ValueError("Balance can't be negative.")

        self.owner = owner
        self.balance = balance 

        BankAccount.total_accounts += 1
        self.account_id = f"ACC-{BankAccount.total_accounts:03d}"

    def deposit (self, amount):
        if amount <= 0:
            raise ValueError("Amount isn't positive.") 

        self.balance += amount
    
    def withdraw (self, amount):
        if amount <= 0:
            raise ValueError("Amount isn't positive.") 
        elif amount > self.balance:
            raise ValueError("Not enough amount on balance")

        self.balance -= amount

    def summary (self):
        return f"[{self.account_id}] {self.owner}: {self.balance:.2f} AZN"

# END HERE


In [2]:
# DO NOT TOUCH

BankAccount.total_accounts = 0
aysel = BankAccount("Aysel", 500)
rashad = BankAccount("Rashad", 0)

aysel.deposit(250)
aysel.withdraw(100)
rashad.deposit(80)

assert BankAccount.bank_name == "AI Academy Bank"
assert BankAccount.total_accounts == 2
assert aysel.account_id == "ACC-001"
assert rashad.account_id == "ACC-002"
assert aysel.balance == 650
assert rashad.balance == 80
assert aysel.summary() == "[ACC-001] Aysel: 650.00 AZN"
assert rashad.summary() == "[ACC-002] Rashad: 80.00 AZN"

for bad in [0, -10]:
    try:
        aysel.deposit(bad)
        raise AssertionError("deposit() must reject non-positive amounts")
    except ValueError:
        pass

for bad in [0, -10]:
    try:
        aysel.withdraw(bad)
        raise AssertionError("withdraw() must reject non-positive amounts")
    except ValueError:
        pass

try:
    rashad.withdraw(1000)
    raise AssertionError("withdraw() must reject insufficient funds")
except ValueError:
    pass

try:
    BankAccount("Invalid", -1)
    raise AssertionError("Negative starting balance must be rejected")
except ValueError:
    pass

print(aysel.summary())
print(rashad.summary())
print("Bank:", BankAccount.bank_name)
print("Total accounts:", BankAccount.total_accounts)
print("Task 1 tests passed.")


[ACC-001] Aysel: 650.00 AZN
[ACC-002] Rashad: 80.00 AZN
Bank: AI Academy Bank
Total accounts: 2
Task 1 tests passed.


## Task 2 — Product with Validated Price and VAT

Make an invalid price impossible to set and expose computed values that should not be overwritten directly.

### Requirements
1. Class attribute `VAT_RATE = 0.18`.
2. `__init__(self, name, price, stock=0)` assigns the price through the property.
3. `price` property: raise `TypeError` for non-numbers and `ValueError` for values `<= 0`.
4. Read-only `price_with_vat`, rounded to 2 decimals.
5. Read-only `in_stock`, `True` when `stock > 0`.
6. `sell(quantity=1)` decreases stock and raises `ValueError` if there is not enough stock.
7. Assignment to `price_with_vat` must fail because it is read-only.


In [3]:
# START WRITING HERE

class Product:
    VAT_RATE = 0.18

    def __init__(self, name, price, stock = 0):
      self.name = name
      self.price = price
      self.stock = stock

    @property
    def price(self):
      return self._price

    @price.setter
    def price(self, value):
      if not isinstance(value, (int, float)):
        raise TypeError("Value isn't number")
      elif value <= 0:
        raise ValueError("Value isn't positive")

      self._price = value

    @property
    def price_with_vat(self):
      return round(self._price * (1 + Product.VAT_RATE), 2)

    @property
    def in_stock(self):
      return True if self.stock > 0 else False

    def sell(self, quantity = 1):
      if quantity > self.stock:
        raise ValueError("Not enough in stock")

      self.stock -= quantity

# END HERE


In [4]:
# DO NOT TOUCH

p = Product("Mechanical keyboard", 120.0, stock=2)
assert Product.VAT_RATE == 0.18
assert p.name == "Mechanical keyboard"
assert p.price == 120.0
assert p.price_with_vat == 141.6
assert p.in_stock is True

p.price = 99.99
assert p.price == 99.99
assert p.price_with_vat == 117.99

p.sell()
assert p.stock == 1 and p.in_stock is True
p.sell()
assert p.stock == 0 and p.in_stock is False

try:
    p.sell()
    raise AssertionError("sell() must reject insufficient stock")
except ValueError:
    pass

for bad in [0, -5]:
    try:
        p.price = bad
        raise AssertionError("price must reject values <= 0")
    except ValueError:
        pass

for bad in ["100", None, [100]]:
    try:
        p.price = bad
        raise AssertionError("price must reject non-numeric values")
    except TypeError:
        pass

try:
    p.price_with_vat = 10
    raise AssertionError("price_with_vat must be read-only")
except AttributeError:
    pass

print("Task 2 tests passed.")


Task 2 tests passed.


## Task 3 — Money

Create a value object that behaves like a number but refuses to mix currencies.

### Requirements
1. `__init__(self, amount, currency="AZN")`: round amount to 2 decimals and uppercase currency.
2. `__repr__` → `Money(19.99, 'AZN')`.
3. `__str__` → `19.99 AZN`.
4. Implement `__eq__`, `__lt__`, `__add__`; all must raise `TypeError` when currencies differ.
5. `__mul__` multiplies by a plain number only.
6. Sorting and `max()` must work on same-currency objects.

**Hint:** a helper such as `_check(other)` can avoid repeated currency-checking logic.


In [5]:
# START WRITING HERE

class Money:
    def __init__(self, amount, currency = 'AZN'):
        self.amount = amount
        self.currency = currency.upper()

    def __repr__(self):
        return f"Money({self.amount:.2f}, '{self.currency}')"

    def __str__(self):
        return f"{self.amount:.2f} {self.currency}"
    
    def _check(self, other):
        if not isinstance(other, Money):
            raise TypeError("Not money")
        
        if self.currency != other.currency:
            raise TypeError("Different currencies")

        return True

    def __eq__ (self, other):
        check = self._check(other)
        if check:
            return self.amount == other.amount

    def __lt__ (self, other):
        check = self._check(other)
        if check:
            return self.amount < other.amount

    def __add__ (self, other):
        check = self._check(other)
        if check:
            return Money(self.amount + other.amount, self.currency)

    def __mul__ (self, number):
        if not isinstance(number, (int, float)):
            raise TypeError("Only number multiplies")
        
        return Money(self.amount * number, self.currency)

# END HERE


In [6]:
# DO NOT TOUCH

a = Money(19.99)
b = Money(5.50, "azn")

assert a.amount == 19.99
assert a.currency == "AZN"
assert repr(a) == "Money(19.99, 'AZN')"
assert str(a) == "19.99 AZN"

result = a + b
assert isinstance(result, Money)
assert result.amount == 25.49
assert str(result) == "25.49 AZN"

triple = a * 3
assert isinstance(triple, Money)
assert triple.amount == 59.97
assert str(triple) == "59.97 AZN"

assert a == Money(19.99)
assert b < a

values = [Money(12), Money(3.5), Money(40), Money(7.25)]
ordered = sorted(values)
assert [str(x) for x in ordered] == ["3.50 AZN", "7.25 AZN", "12.00 AZN", "40.00 AZN"]
assert str(max(values)) == "40.00 AZN"

for operation in [
    lambda: a + Money(2, "USD"),
    lambda: a == Money(19.99, "USD"),
    lambda: a < Money(30, "USD"),
]:
    try:
        operation()
        raise AssertionError("Different currencies must raise TypeError")
    except TypeError:
        pass

try:
    a * "3"
    raise AssertionError("__mul__ must reject non-numeric multipliers")
except TypeError:
    pass

print("Task 3 tests passed.")


Task 3 tests passed.


## Task 4 — Payroll Hierarchy

Create three classes with one shared interface and different salary rules.

### Requirements
1. `Employee(name, base_salary)` with `pay()` returning base salary.
2. `Employee.__str__()` must use `self.__class__.__name__`, e.g. `Developer Tural: 2600.00 AZN`.
3. `Developer(name, base_salary, shipped_features=0)`: base salary + 150 per shipped feature.
4. `Manager(name, base_salary, team_size=0)`: base salary increased by 5% per team member.
5. Both subclasses must call `super().__init__()` and reuse `super().pay()`.
6. Support looping through mixed employee types, `isinstance(...)`, and `Developer.__mro__`.


In [8]:
# START WRITING HERE

class Employee:
    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary
    
    def pay(self):
        return self.base_salary
    
    def __str__(self):
        return f"{self.__class__.__name__} {self.name}: {self.pay():.2f} AZN"


class Developer(Employee):
    def __init__(self, name, base_salary, shipped_features = 0):
        super().__init__(name, base_salary)
        self.shipped_features = shipped_features
    
    def pay(self):
        return super().pay() + 150 * self.shipped_features


class Manager(Employee):
    def __init__(self, name, base_salary, team_size = 0):
        super().__init__(name, base_salary)
        self.team_size = team_size

    def pay(self):
        return super().pay() * (1 + 0.05 * self.team_size)

# END HERE


In [9]:
# DO NOT TOUCH

staff = [
    Employee("Nigar", 1500),
    Developer("Tural", 2000, shipped_features=4),
    Manager("Leyla", 2500, team_size=6),
]

assert staff[0].pay() == 1500
assert staff[1].pay() == 2600
assert staff[2].pay() == 3250
assert str(staff[0]) == "Employee Nigar: 1500.00 AZN"
assert str(staff[1]) == "Developer Tural: 2600.00 AZN"
assert str(staff[2]) == "Manager Leyla: 3250.00 AZN"
assert isinstance(staff[1], Employee) is True
assert [cls.__name__ for cls in Developer.__mro__] == ["Developer", "Employee", "object"]
assert sum(person.pay() for person in staff) == 7350

for person in staff:
    print(person)
print("Payroll total:", sum(person.pay() for person in staff))
print("Task 4 tests passed.")


Employee Nigar: 1500.00 AZN
Developer Tural: 2600.00 AZN
Manager Leyla: 3250.00 AZN
Payroll total: 7350.0
Task 4 tests passed.


## Task 5 — Shape Hierarchy

Create one abstract contract with three concrete implementations.

### Requirements
1. `Shape(ABC)` with abstract `area()` and `perimeter()`.
2. `Shape.describe()` is concrete and shows both values to 2 decimals.
3. `Circle(radius)` rejects a non-positive radius.
4. `Rectangle(width, height)`.
5. `Triangle(a, b, c)` rejects invalid side lengths and uses Heron's formula.
6. Test `[Circle(3), Rectangle(4, 5), Triangle(3, 4, 5)]`.
7. Find the largest shape by area and total area.
8. Instantiating `Shape()` must raise `TypeError`.

Heron's formula: `s = perimeter / 2`, then `sqrt(s*(s-a)*(s-b)*(s-c))`.


In [1]:
# DO NOT TOUCH
from abc import ABC, abstractmethod
import math


In [29]:
# START WRITING HERE

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

    def describe(self):
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, radius):
        if radius <= 0:
            raise ValueError("Radius isn't positive")
        self.radius = radius
    
    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius 


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height 

    def perimeter(self):
        return 2 * (self.width + self.height)  


class Triangle(Shape):
    def __init__(self, a, b, c):
        sides = sorted([a, b, c])
        if sides[0] <= 0 or sides[0] + sides[1] <= sides[2]:
            raise ValueError("Invalid side lengths")
        self.a = a
        self.b = b
        self.c = c

    def perimeter(self):
        return self.a + self.b + self.c

    def area(self):
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

# END HERE


In [30]:
# DO NOT TOUCH

shapes = [Circle(3), Rectangle(4, 5), Triangle(3, 4, 5)]
assert round(shapes[0].area(), 2) == 28.27
assert round(shapes[0].perimeter(), 2) == 18.85
assert round(shapes[1].area(), 2) == 20.00
assert round(shapes[1].perimeter(), 2) == 18.00
assert round(shapes[2].area(), 2) == 6.00
assert round(shapes[2].perimeter(), 2) == 12.00
assert shapes[0].describe() == "Circle: area=28.27, perimeter=18.85"
assert shapes[1].describe() == "Rectangle: area=20.00, perimeter=18.00"
assert shapes[2].describe() == "Triangle: area=6.00, perimeter=12.00"

largest = max(shapes, key=lambda s: s.area())
assert largest.__class__.__name__ == "Circle"
assert round(sum(s.area() for s in shapes), 2) == 54.27

try:
    Circle(0)
    raise AssertionError("Circle must reject non-positive radius")
except ValueError:
    pass

for sides in [(1, 2, 3), (1, 1, 3)]:
    try:
        Triangle(*sides)
        raise AssertionError("Triangle must enforce triangle inequality")
    except ValueError:
        pass

try:
    Shape()
    raise AssertionError("Shape must be abstract")
except TypeError:
    pass

for shape in shapes:
    print(shape.describe())
print("Largest:", largest.__class__.__name__)
print("Total area:", f"{sum(s.area() for s in shapes):.2f}")
print("Task 5 tests passed.")


Circle: area=28.27, perimeter=18.85
Rectangle: area=20.00, perimeter=18.00
Triangle: area=6.00, perimeter=12.00
Largest: Circle
Total area: 54.27
Task 5 tests passed.


## Task 6 — Playlist and Song

Build a container class and make it behave naturally with Python.

### Requirements
1. `Song(title, artist, seconds)` with `__str__()` like `Debussy - Clair de Lune (5:05)`.
2. A helper formats seconds as `m:ss`.
3. `Playlist(name)` holds a private list of songs.
4. `add(song)` rejects non-`Song` values and returns `self`.
5. `remove(title)` is case-insensitive and returns `True` or `False`.
6. Implement `total_duration()` and `by_artist(artist)`.
7. Implement `__len__`, `__iter__`, `__getitem__`.
8. `Playlist.__str__()` should look like `Focus (3 songs, 14:25)`.

> **Source note:** the homework's printed expected output says `After removal: Focus (2 songs, 9:20)`, but the listed durations for Debussy (5:05) and Einaudi (5:50) add to **10:55**. The test cell uses 10:55 so it matches the actual durations given in the task.


In [31]:
# START WRITING HERE

def format_duration(seconds):
        minutes, secs = divmod(seconds, 60)
        return f"{minutes}:{secs:02d}"


class Song:
    def __init__(self, title, artist, seconds):
        self.title = title
        self.artist = artist
        self.seconds = seconds
    
    def __str__(self):
        return f"{self.artist} - {self.title} ({format_duration(self.seconds)})"


class Playlist:
    def __init__(self, name):
        self.name = name
        self._songs = []
    
    def add(self, song):
        if not isinstance(song, Song):
            raise TypeError("Only songs can be added")
        self._songs.append(song)
        return self
    
    def remove(self, title):
        for index, song in enumerate(self._songs):
            if song.title.lower() == title.lower():
                self._songs.pop(index)
                return True
        return False

    def total_duration(self):
        return sum(song.seconds for song in self._songs)

    def by_artist(self, artist):
        return [song for song in self._songs if song.artist == artist]
    
    def __len__(self):
        return len(self._songs)

    def __iter__(self):
        return iter(self._songs)

    def __getitem__(self, index):
        return self._songs[index]

    def __str__(self):
        song_count = len(self._songs)
        duration_str = format_duration(self.total_duration())
        return f"{self.name} ({song_count} songs, {duration_str})"
    
# END HERE


In [32]:
# DO NOT TOUCH

s1 = Song("Clair de Lune", "Debussy", 305)
s2 = Song("Gymnopedie No.1", "Satie", 210)
s3 = Song("Nuvole Bianche", "Einaudi", 350)

playlist = Playlist("Focus")
returned = playlist.add(s1).add(s2).add(s3)
assert returned is playlist
assert len(playlist) == 3
assert playlist[0] is s1
assert list(iter(playlist)) == [s1, s2, s3]
assert str(s1) == "Debussy - Clair de Lune (5:05)"
assert str(s2) == "Satie - Gymnopedie No.1 (3:30)"
assert str(s3) == "Einaudi - Nuvole Bianche (5:50)"
assert playlist.total_duration() == 865
assert str(playlist) == "Focus (3 songs, 14:25)"
assert [song.title for song in playlist.by_artist("Satie")] == ["Gymnopedie No.1"]
assert playlist.remove("gymnopedie no.1") is True
assert len(playlist) == 2
assert str(playlist) == "Focus (2 songs, 10:55)"
assert playlist.remove("does not exist") is False

try:
    playlist.add("not a song")
    raise AssertionError("add() must reject non-Song objects")
except TypeError:
    pass

print("Task 6 tests passed.")


Task 6 tests passed.


## Task 7 — User with Three Ways to Build It

Create one class with several construction methods and one shared validation rule.

### Requirements
1. `User(username, email, age)` validates the email using the static method, raises `ValueError` if invalid, and increments class attribute `user_count`.
2. `is_valid_email(email)` as `@staticmethod`: requires an `@` and a dot in the part after it.
3. Three `@classmethod` constructors: `from_csv(...)`, `from_dict(...)`, and `guest()`.
4. `__repr__()` example: `User('aysel', 'aysel@ufaz.az', 21)`.
5. Every successfully created user counts toward `user_count`.


In [25]:
# START WRITING HERE

class User:
    user_count = 0

    def __init__(self, username, email, age):
        if not self.is_valid_email(email):
            raise ValueError("Invalid email")
        self.username = username
        self.email = email
        self.age = age
        User.user_count += 1

    @staticmethod
    def is_valid_email(email):
        if "@" not in email:
            return False
        local, _, domain = email.partition("@")
        if not local or not domain:
            return False
        return "." in domain
    
    @classmethod
    def from_csv(cls, csv_line):
        username, email, age = csv_line.split(",")
        return cls(username.strip(), email.strip(), int(age.strip()))

    @classmethod
    def from_dict(cls, data):
        return cls(data["username"], data["email"], data["age"])

    @classmethod
    def guest(cls):
        return cls("guest", "guest@example.com", 0)

    def __repr__(self):
        return f"User('{self.username}', '{self.email}', {self.age})"

# END HERE


In [27]:
# DO NOT TOUCH

User.user_count = 0
u1 = User("aysel", "aysel@ufaz.az", 21)
u2 = User.from_csv("tural, tural@socar.az, 25")
u3 = User.from_dict({"username": "leyla", "email": "leyla@mail.com", "age": 30})
u4 = User.guest()

assert repr(u1) == "User('aysel', 'aysel@ufaz.az', 21)"
assert repr(u2) == "User('tural', 'tural@socar.az', 25)"
assert repr(u3) == "User('leyla', 'leyla@mail.com', 30)"
assert repr(u4) == "User('guest', 'guest@example.com', 0)"
assert User.user_count == 4
assert User.is_valid_email("a@b.com") is True
assert User.is_valid_email("broken@mail") is False
assert User.is_valid_email("not-an-email") is False

try:
    User("bad", "not-an-email", 20)
    raise AssertionError("Invalid email must raise ValueError")
except ValueError:
    pass

print(u1)
print(u2)
print(u3)
print(u4)
print("User count:", User.user_count)
print("Task 7 tests passed.")


User('aysel', 'aysel@ufaz.az', 21)
User('tural', 'tural@socar.az', 25)
User('leyla', 'leyla@mail.com', 30)
User('guest', 'guest@example.com', 0)
User count: 4
Task 7 tests passed.


## Task 8 — Library, LibraryItem, Book, DVD, Member

Build a small lending library using abstraction, inheritance, a property, composition, container methods, and exceptions.

### `LibraryItem(ABC)`
- `item_id`, `title`, `year`
- private `_borrowed_by`, initially `None`
- read-only `is_available` property
- abstract `loan_days()`
- `__str__()` like `B1 | Clean Code (2008) - available`

### `Book`
- adds `author`, `pages`
- more than 400 pages → 21 loan days; otherwise 14

### `DVD`
- adds `minutes`
- always 3 loan days

### `Member`
- borrowed-items list
- class attribute `MAX_ITEMS = 2`
- read-only `can_borrow` property

### `Library`
- stores items and members in dictionaries
- `add_item(...)` and `register(...)` return `self`
- `borrow(member_name, item_id)` rejects unavailable items or a member at the limit, links both sides, and returns a confirmation string
- `give_back(...)` reverses the relationship and returns a confirmation string
- `available_items()`
- `__len__` and `__iter__` over items

### Scenario
Use 2 books, 1 DVD, and 2 members. Aysel borrows two items and then hits her limit. Tural then fails to borrow a book that is already out.


In [ ]:
# DO NOT TOUCH
from abc import ABC, abstractmethod


In [48]:
# START WRITING HERE

class LibraryItem(ABC):
    def __init__(self, item_id, title, year):
        self.item_id = item_id
        self.title = title
        self.year = year
        self._borrowed_by = None

    @property
    def is_available(self):
        return self._borrowed_by is None

    @abstractmethod
    def loan_days(self):
        pass

    def __str__(self):
        status = "available" if self.is_available else f"with {self._borrowed_by}"
        return f"{self.item_id} | {self.title} ({self.year}) - {status}"


class Book(LibraryItem):
    def __init__(self, item_id, title, year, author, pages):
        super().__init__(item_id, title, year)
        self.author = author
        self.pages = pages

    def loan_days(self):
        return 21 if self.pages > 400 else 14


class DVD(LibraryItem):
    def __init__(self, item_id, title, year, minutes):
        super().__init__(item_id, title, year)
        self.minutes = minutes

    def loan_days(self):
        return 3


class Member:
    MAX_ITEMS = 2

    def __init__(self, name):
        self.name = name
        self._borrowed_items = []

    @property
    def can_borrow(self):
        return len(self._borrowed_items) < Member.MAX_ITEMS


class Library:
    def __init__(self, name):
        self.name = name
        self._items = {}
        self._members = {}

    def add_item(self, item):
        self._items[item.item_id] = item
        return self

    def register(self, member):
        self._members[member.name] = member
        return self

    def borrow(self, member_name, item_id):
        item = self._items.get(item_id)
        if item is None:
            raise KeyError(f"No such item: {item_id}")
        member = self._members.get(member_name)

        if member is None:
            raise KeyError(f"No such member: {member_name}")

        if not item.is_available:
            raise ValueError(f"Item {item_id} is not available")
            
        if not member.can_borrow:
            raise ValueError(f"{member_name} has reached the borrowing limit")

        item._borrowed_by = member_name
        member._borrowed_items.append(item)

        return f"{member_name} borrowed '{item.title}' for {item.loan_days()} days"

    def give_back(self, member_name, item_id):
        item = self._items.get(item_id)
        if item is None:
            raise KeyError(f"No such item: {item_id}")
        member = self._members.get(member_name)
        if member is None:
            raise KeyError(f"No such member: {member_name}")

        item._borrowed_by = None
        if item in member._borrowed_items:
            member._borrowed_items.remove(item)

        return f"{member_name} returned '{item.title}'"

    def available_items(self):
        return [item for item in self._items.values() if item.is_available]

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items.values())

# END HERE


In [49]:
# DO NOT TOUCH

book1 = Book("B1", "Clean Code", 2008, "Robert C. Martin", 464)
book2 = Book("B2", "Fluent Python", 2022, "Luciano Ramalho", 1014)
dvd1 = DVD("D1", "Interstellar", 2014, 169)
aysel = Member("Aysel")
tural = Member("Tural")

library = Library("AI Academy Library")
returned = (library.add_item(book1).add_item(book2).add_item(dvd1).register(aysel).register(tural))
assert returned is library
assert len(library) == 3
assert list(iter(library)) == [book1, book2, dvd1]
assert book1.is_available is True
assert book2.is_available is True
assert dvd1.is_available is True
assert book1.loan_days() == 21
assert book2.loan_days() == 21
assert dvd1.loan_days() == 3
assert aysel.can_borrow is True
assert Member.MAX_ITEMS == 2

msg1 = library.borrow("Aysel", "B1")
msg2 = library.borrow("Aysel", "D1")
assert msg1 == "Aysel borrowed 'Clean Code' for 21 days"
assert msg2 == "Aysel borrowed 'Interstellar' for 3 days"
assert book1.is_available is False
assert dvd1.is_available is False
assert aysel.can_borrow is False

try:
    library.borrow("Aysel", "B2")
    raise AssertionError("Member borrowing limit must be enforced")
except ValueError:
    pass

try:
    library.borrow("Tural", "B1")
    raise AssertionError("Already borrowed item must be rejected")
except ValueError:
    pass

available_titles = [item.title for item in library.available_items()]
assert available_titles == ["Fluent Python"]

return_msg = library.give_back("Aysel", "D1")
assert return_msg == "Aysel returned 'Interstellar'"
assert dvd1.is_available is True
assert aysel.can_borrow is True
assert str(book1) == "B1 | Clean Code (2008) - with Aysel"
assert str(book2) == "B2 | Fluent Python (2022) - available"
assert str(dvd1) == "D1 | Interstellar (2014) - available"

print(f"{library.name}: {len(library)} items")
print(msg1)
print(msg2)
print("Available now:", available_titles)
print(return_msg)
print("--- catalogue ---")
for item in library:
    print(item)
print("Task 8 tests passed.")


AI Academy Library: 3 items
Aysel borrowed 'Clean Code' for 21 days
Aysel borrowed 'Interstellar' for 3 days
Available now: ['Fluent Python']
Aysel returned 'Interstellar'
--- catalogue ---
B1 | Clean Code (2008) - with Aysel
B2 | Fluent Python (2022) - available
D1 | Interstellar (2014) - available
Task 8 tests passed.


## Going Further — Optional

- Add a `Magazine` item type without changing `Library`.
- Replace borrowing-related `ValueError`s with custom exception classes deriving from `LibraryError(Exception)`.
- Add due dates using `datetime`.
- Add a fine of `0.50 AZN` per late day.
